# AlexNet Benchmark

use PY310, py311 not support torch.compile, py39 not support libnvrtc.so compatibility

In [1]:
model_name = "vit-torch"

import torch
from torch import nn
from torchvision import models
#import torch_mlir
import numpy as np
import iree
import iree.compiler
import iree.runtime

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll


In [2]:
    
def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=5):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

## Experimental

In [3]:
import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cpu:0")
model = models.vit_l_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()

### PyTorch (Baseline)

In [4]:
for bs in range(1,16):
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    
    try:
        baseline_f = torch_model_benchmark(model, 
                                 [image], 
                                 device='cpu',
                                 warmups=0,
                                 repetitions=5, 
                                 measure_count=11)
        baseline_f = np.mean(baseline_f)
        baseline_b = torch_model_benchmark(torch.autograd.grad, 
                                 [output, [image], grad], 
                                 device='cpu',
                                 warmups=0,
                                 repetitions=5, 
                                 measure_count=11)
        baseline_b = np.mean(baseline_b)
        df = pd.concat([df, get_dataframe(baseline_f, baseline_b, "Torch Native at batch-size = {}".format(bs))])
    except Exception as e:
        print(f"处理模型时出错，批量大小 {bs}: {e}")
    print(df)

        time      pass                            item
0  17.154511   Forward  Torch Native at batch-size = 1
0  28.462999  Backward  Torch Native at batch-size = 1
0  45.617509      Full  Torch Native at batch-size = 1
        time      pass                            item
0  17.154511   Forward  Torch Native at batch-size = 1
0  28.462999  Backward  Torch Native at batch-size = 1
0  45.617509      Full  Torch Native at batch-size = 1
0  29.307227   Forward  Torch Native at batch-size = 2
0  44.112997  Backward  Torch Native at batch-size = 2
0  73.420225      Full  Torch Native at batch-size = 2
         time      pass                            item
0   17.154511   Forward  Torch Native at batch-size = 1
0   28.462999  Backward  Torch Native at batch-size = 1
0   45.617509      Full  Torch Native at batch-size = 1
0   29.307227   Forward  Torch Native at batch-size = 2
0   44.112997  Backward  Torch Native at batch-size = 2
0   73.420225      Full  Torch Native at batch-size = 2
0  

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 7.78 GiB total capacity; 6.96 GiB already allocated; 3.94 MiB free; 7.56 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
df.style.hide(axis="index")

In [ ]:
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")

In [ ]:
forward = df[df["pass"] == "Forward"]
forward["acceleration"] = baseline_f / forward["time"]
forward

In [ ]:
backward = df[df["pass"] == "Backward"]
backward["acceleration"] = baseline_b / backward["time"]
backward

In [ ]:
full = df[df["pass"] == "Full"]
full["acceleration"] = (baseline_b + baseline_f) / full["time"]
full

In [ ]:
df = pd.concat([forward, backward, full])
df.to_csv(f"{model_name}.csv")

sns.barplot(df, x="pass", y="acceleration", hue="item")
plt.xlabel("Pass")
plt.ylabel("Acceleration")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-acceleration.png")

In [ ]:
df.to_csv('output-native-cpu.csv')
df.transpose().to_csv('output2-native-cpu.csv')